# Evaluation Report for the paper "_dlinear_: Enhancing SMT Solvers with Floating-Point Exact LP Solvers"

---

## 1. Executive Summary

This report presents a comprehensive comparison between three approaches for integrating linear programming solvers into the baseline [cvc5](https://github.com/cvc5/cvc5) SMT solver:

1. **cvc5 baseline**: The default configuration without any external LP solver
2. **cvc5+GLPK**: Using [GLPK](https://www.gnu.org/software/glpk/) as an external floating-point LP solver
3. **_dlinear_** (our proposed approach): Using floating-point exact LP solvers, specifically [SoPlex](https://github.com/scipopt/soplex) and [qsoptex](https://github.com/TendTo/qsopt-ex)

The results shown are obtained by running `./run.sh [suite] [limit] [timeout]`, where `[suite]` is the benchmark suite name (default: `smoke`) and `[limit]` is the number of instances to run (default: `6`) and `[timeout]` is the time limit per instance in seconds (default: `21000`).
The metrics collected for each configuration are read from CSV files in `results-[suite]/`.

For more details on the contributions, please refer to the paper.

---

In [ ]:
from IPython.display import Markdown as md
import pandas as pd
from functools import reduce
from utils import (
    compare_unique_solved_instances,
    build_markdown_comparison_matrix,
    plot_performance_profiles,
    sanitize,
    difficulty_analysis,
    print_stats,
    SolverResult,
    external_solver_impact,
)
from itertools import product
import os

DROP_UNSOLVED = False
SOLVERS = ("soplex", "qsoptex")
ITERATIONS = (0, 100, 200,  300)
MODES = ("", "_strict", "_delta1e+30") 

def get_tot_results_compare(result: SolverResult, iterations_filter: list[tuple[int, pd.DataFrame]]) -> int:
    header = ""
    middle = ""
    row = ""
    for iterations, instances in iterations_filter:
        results = result.dataframe[result.dataframe.index.isin(instances.index)]
        results = results[results[result.result_key].isin(["sat", "unsat"])]
        not_solved = len(instances) - len(results)
        header += f"| $n={iterations}$ solved | $n={iterations}$ unsolved " if iterations != -1 else "| Solved | Unsolved "
        middle += f"| --- | --- "
        len_instances = max(len(instances), 1)
        row += f"| {len(results)} / {len(instances)} ({len(results) / len_instances * 100:.1f}%) | {not_solved} / {len(instances)} ({not_solved / len_instances * 100:.1f}%) "
    return f"""
#### {result.solver_name} results

| Solver               {header} |
| -------------------- {middle} |
| {result.solver_name} {row} |
"""

run_name = os.getenv("RUN_NAME", "smoke")
local_limit = int(os.getenv("LOCAL_LIMIT", -1))
all_instances = pd.read_csv(f"/work/instances/{run_name}.csv").set_index("file")
if local_limit > 0:
    all_instances = all_instances.head(local_limit)

instances_dict = {
    0: all_instances.copy(),
    100: all_instances.copy(),
    200: all_instances.copy(),
    300: all_instances.copy(),
}

def in_n(df: pd.DataFrame, n: int) -> pd.DataFrame:
    return df[df.index.isin(instances_dict[n].index)]

results: dict[str, SolverResult] = {}
for solver, iteration, m in product(SOLVERS, ITERATIONS, MODES):
    key = f"{solver}_i{iteration}{m}"
    file = f"results/{key}.csv"
    solver_id = solver[0].upper()
    if os.path.exists(file):
        df = pd.read_csv(file).set_index("file")
        instances_dict[iteration] = instances_dict[iteration].loc[~instances_dict[iteration].index.isin(df[df["theory::arith::z::arith::relax::calls"].astype(int) == 0].index)]
        df = df[df["theory::arith::z::arith::relax::calls"].astype(int) > 0]
        if DROP_UNSOLVED:
            df = df[df[f"result{solver_id}"].isin(["sat", "unsat"])]
        if m == "_strict":
            mode_str = "t"
        elif "_delta" in m:
            mode_str = r"\delta"
        else:
            mode_str = r"\varepsilon"
        results[key] = SolverResult(solver_name=fr"$ \textbf{{d}}_{{{solver[:2].lower()},{iteration}}}^{mode_str} $", dataframe=df, solver_id=solver_id, iterations=iteration)

for key in results:
    results[key] = results[key].replace_df(sanitize(results[key].dataframe))

others = ("glpk_i100", "glpk_i200", "glpk_i300", "yices", "z3", "cvc5", )
others_results: dict[str, SolverResult] = {}
for others in others:
    path = f"results/{others}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path).set_index("file")
        if DROP_UNSOLVED:
            df = df[df[f"result{others[0].upper()}"].isin(["sat", "unsat"])]
        iterations = f"_{{{others.split('_')[-1][1:]}}}" if "glpk" in others else ""
        df = df[df["theory::arith::z::arith::relax::calls"].astype(int) > 0] if "glpk" in others else df
        others_results[others] = SolverResult(solver_name=fr"$ \textbf{{{others[0].lower()}}}{iterations} $", dataframe=df, solver_id=others[0].upper(), iterations=int(iterations[2:-1]) if iterations else -1)

## 2. Overall Benchmark Performance

### 2.1 Problems Solved

This section summarizes the number of instances solved by each approach across the selected benchmark suite.
Note that we use the [SMT-LIB](https://smt-lib.org/benchmarks.shtml) benchmarks for our evaluation.
More precisely, we use the [QF_LRA](https://smt-lib.org/logics-all.shtml#QF_LRA) theory benchmarks from the [SMT-LIB release 2025 of non-incremental benchmarks](https://zenodo.org/records/16740866).

In [ ]:
md(f"""### Instances

The results have been collected on a total of **{len(all_instances)}** instances.
""")

### 2.2 Solver Notation

When abbreviating the solvers, we use the following conventions:

- $\textbf{c}$, $\textbf{y}$, and $\textbf{z}$ for _cvc5_, _yices_, and _z3_, respectively
- $\textbf{g}_n$ for _glpk_, where $n \in \{0, 100, 200, 300\}$ is the number of pivots performed by the rational simplex implementation before calling the external LP solver
- $\textbf{d}^m_{s,n}$ for _dlinear_ in complete mode, where:
  - $s \in \{\text{so}, \text{qs}\}$ indicates the external LP solver (_soplex_ or _qsoptex_)
  - $m \in \{\varepsilon, t\}$ indicates how strict inequalities are handled ($\varepsilon$ perturbation on strict constraints or use of a strict variable $t$)
  - $n$ is the pivot threshold as described above
- $\textbf{d}^{\delta}_{s, n}$ for _dlinear_ in $\delta$-complete mode, where $s$ and $n$ are as described above

See _Section 6_ of the paper for more details.

In [ ]:
md(
    fr"""### 2.3 Loading _dlinear_ results
   
In the following table, we summarize the loaded results for each solver configuration. 
Each row corresponds to a specific configuration of our _dlinear_ approach, with different pivot thresholds (100, 200, 300) and modes (epsilon, strict, delta). 
The "Loaded Instances" column indicates how many benchmark instances were successfully loaded for each configuration.
Higher pivot thresholds are expected to have less loaded instances, as we only consider those that performed at least one call to the external LP solver.

The _Mode_ column indicates how constraints with strict inequalities (i.e., $<, >$) are handled.
The possible modes are:
   
- $\varepsilon$: The default mode, which uses a small perturbation $\varepsilon$ on the right-hand side of the constraints to ensure strict inequalities are satisfied.
- $t$: A mode that adds and additional strict variable $t$ to constraint containing strict inequalities, and then tries to maximize $t$ to ensure that the constraints are satisfied with a positive margin.
- $\delta$: $\delta$-complete mode, which uses a user-defined $\delta$ value to relax the constraints, effectively transforming strict inequalities into non-strict ones.

Additional details can be found in _Sections 4.2_ of the paper.
   
| Solver | Mode |  Pivot Threshold | Available Instances | Solved Instances | Unsolved Instances |
|--------|-----------------|------|-------------------| -----------------|-------------------|
{"".join(f"| {res.solver_name} | {res.mode}  | {res.iterations} | {len(in_n(all_instances, res.iterations))} | {len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])])} ({len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])]) / len(in_n(all_instances, res.iterations)) * 100:.1f}%) | {len(in_n(all_instances, res.iterations)) - len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])])} ({(len(in_n(all_instances, res.iterations)) - len(res.dataframe[res.dataframe['result'].isin(['sat', 'unsat'])])) / len(in_n(all_instances, res.iterations)) * 100:.1f}%) |\n" for res in results.values())}
"""
)

### 2.4 External Solver Impact

Calls to the external simplex solver can significantly affect the overall performance of the SMT solver, particularly when precision boosting and iterative refinement techniques are costly, which is common for numerically challenging instances. Here, we analyze the impact of:

- **Precision boosting**: The number $p_n$ of calls that required $n$ bits of precision
- **Iterative refinements**: The number $r_i$ of calls that required $i$ refinement iterations

This analysis helps quantify the overhead introduced by exact arithmetic techniques.

In [ ]:
md(external_solver_impact(
    solvers_analysis=results.values(),
    filename="summary_exact_solver_impact"
))

## 3. Difficulty and Performance Analysis

### 3.1 Difficulty Analysis

We analyze the distribution of solving times across benchmark instances, dividing them into different buckets based on the time taken to solve each instance.

In [ ]:
md(difficulty_analysis(
    solvers_analysis=[
        others_results.get("cvc5", SolverResult.empty()).apply_filter(lambda df: in_n(df, 100)),
        others_results.get("glpk_i100", SolverResult.empty()),
        results.get("soplex_i100", SolverResult.empty()),
        results.get("soplex_i100_strict", SolverResult.empty()),
        # results.get("soplex_i100_delta1e-08"),
        results.get("qsoptex_i100", SolverResult.empty()),
        results.get("qsoptex_i100_strict", SolverResult.empty()),
        # results.get("qsoptex_i100_delta1e-08"),
    ],
    total_count= len(all_instances),
    group_name="instances with $n = 100$ pivots threshold",
))

In [ ]:
md(difficulty_analysis(
    solvers_analysis=[
        others_results.get("cvc5",SolverResult.empty()).apply_filter(lambda df: in_n(df, 200)),
        others_results.get("glpk_i200",SolverResult.empty()),
        results.get("soplex_i200",SolverResult.empty()),
        results.get("soplex_i200_strict",SolverResult.empty()),
        # results.get("soplex_i200_delta1e-08"),
        results.get("qsoptex_i200",SolverResult.empty()),
        results.get("qsoptex_i200_strict",SolverResult.empty()),
        # results.get("qsoptex_i200_delta1e-08"),
    ],
   total_count= len(all_instances),
   group_name="instances with $n = 200$ pivots threshold",
))

In [ ]:
md(difficulty_analysis(
    solvers_analysis=[
        others_results.get("cvc5", SolverResult.empty()).apply_filter(lambda df: in_n(df, 300)),
        others_results.get("glpk_i300", SolverResult.empty()),
        results.get("soplex_i300", SolverResult.empty()),
        results.get("soplex_i300_strict", SolverResult.empty()),
        results.get("qsoptex_i300", SolverResult.empty()),
        results.get("qsoptex_i300_strict", SolverResult.empty()),
    ],
   total_count= len(all_instances),
   group_name="instances with $n = 300$ pivots threshold",
))

### 3.2 Performance Profiles

Performance profiles are a powerful tool for comparing the performance of multiple solvers across a benchmark set.
They highlight the percentage of instances each solver can complete within a given time factor of the best solver for each instance, providing a clear visualization of relative performance.

In [ ]:
def to_new_solver_id(solver_result: SolverResult):
    return f"{solver_result.solver_id}{solver_result.iterations}{'S' if '^t' in solver_result.solver_name.lower() else ''}{'D' if r'\delta' in solver_result.solver_name.lower() else ''}"

cvc5_variants_results = [
    SolverResult(
        dataframe=solver_result.dataframe.reset_index()[
            ["file", f"time{solver_result.solver_id}", f"result{solver_result.solver_id}", *(["options::pivots"] if "options::pivots" in solver_result.dataframe.columns else [])]
        ]
        .rename(
            columns={
                f"time{solver_result.solver_id}": f"time{to_new_solver_id(solver_result)}",
                f"result{solver_result.solver_id}": f"result{to_new_solver_id(solver_result)}",
            }
        )
        .set_index("file"),
        solver_name=solver_result.solver_name,
        solver_id=to_new_solver_id(solver_result),
        iterations=solver_result.iterations,
    )
    for solver_result in (results | others_results).values()
    if solver_result.solver_id in ["C", "S", "Q", "G"]
]

other_profile_results = [
    SolverResult(
        dataframe=solver_result.dataframe.reset_index()[
            ["file", f"time{solver_result.solver_id}", f"result{solver_result.solver_id}"]
        ]
        .rename(
            columns={
                f"time{solver_result.solver_id}": f"time{to_new_solver_id(solver_result)}",
                f"result{solver_result.solver_id}": f"result{to_new_solver_id(solver_result)}",
            }
        )
        .set_index("file"),
        solver_name=solver_result.solver_name,
        solver_id=to_new_solver_id(solver_result),
        iterations=solver_result.iterations,
    )
    for solver_result in others_results.values()
    if solver_result.solver_id in ["Y", "Z"]
]

# Get the set of instances where any of the SoPlex variants makes external simplex calls, and filter all profiles to only those instances
external_cvc5_variants_results: dict[int, list[SolverResult]] = {}
for iteration in ITERATIONS:
        external_cvc5_variants_results[iteration] = [
            result.replace_df(in_n(result.dataframe, iteration).drop(columns=["options::pivots"], errors="ignore"))
            for result in cvc5_variants_results
            if not result.dataframe.empty and ("options::pivots" not in result.dataframe.columns or result.dataframe["options::pivots"].iloc[0] == iteration)
        ]

In [ ]:
if 100 in external_cvc5_variants_results and len(external_cvc5_variants_results[100]) > 1:
    ax, fig = plot_performance_profiles(
        *external_cvc5_variants_results[100],
        accepted_results=["sat", "unsat"],
        result_cols="result",
        metric_cols="time",
        max_tau=1000,
    )
else:
    print("Not enough variants with external calls to plot performance profile for n=100.")

Performance profile for the $n = 100$ pivot threshold.

In [ ]:
if 200 in external_cvc5_variants_results and len(external_cvc5_variants_results[200]) > 1:
    ax, fig = plot_performance_profiles(
        *external_cvc5_variants_results[200],
        accepted_results=["sat", "unsat"],
        result_cols="result",
        metric_cols="time",
        max_tau=1000,
    )
else:
    print("Not enough variants with external calls to plot performance profile for n=200.")

Performance profile for the $n = 200$ pivot threshold.

In [ ]:
if 300 in external_cvc5_variants_results and len(external_cvc5_variants_results[300]) > 1:
    ax, fig = plot_performance_profiles(
        *external_cvc5_variants_results[300],
        accepted_results=["sat", "unsat"],
        result_cols="result",
        metric_cols="time",
        max_tau=1000,
        num_points=1000,
    )
else:
    print("Not enough variants with external calls to plot performance profile for n=300.")

Performance profile for the $n = 300$ pivot threshold.

In [ ]:
if 0 in external_cvc5_variants_results and len(external_cvc5_variants_results[0]) > 1:
    ax, fig = plot_performance_profiles(
        *external_cvc5_variants_results[0],
        accepted_results=["sat", "unsat"],
        result_cols="result",
        metric_cols="time",
        max_tau=1000,
        num_points=1000,
    )
else:
    print("Not enough variants with external calls to plot performance profile for n=0.")

Performance profile for the $n = 0$ pivot threshold.